# Model dataset ETL and site reconciliation

This notebook explains and audits how the 49-site modelling table is assembled, standardised, reconciled and quality checked. It is intentionally separate from the authoritative workbook and never overwrites `data/processed/model_dataset.xlsx`.

In [ ]:
import hashlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    # Keep the audit executable in a plain Python environment.
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)


def find_project_root(start: Path) -> Path:
    # Find the repository root without assuming where Jupyter was launched.
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'processed' / 'model_dataset.xlsx').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'model_dataset.xlsx'
CANDIDATE_OUTPUT_PATH = PROJECT_ROOT / '01_data_preparation' / 'model_dataset_candidate.xlsx'
CLIENT_TARGET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Ped_iRAP_merged.csv'
ROAD_NETWORK_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Road_Network.csv'
AUSRAP_PATH = PROJECT_ROOT / 'data' / 'raw' / 'AusRAP Data - Words - MSDS.xlsx'
ARCHIVED_SURVEY_ROOT = None  # Set an approved external archive path only when raw reconstruction is required.

def sha256_file(path):
    # Hash the authoritative workbook so this audit can prove it was not changed.
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

MODEL_SHA256_BEFORE = sha256_file(MODEL_PATH)

print(f'Project root: {PROJECT_ROOT}')
print(f'Authoritative workbook: {MODEL_PATH}')
print(f'Candidate output: {CANDIDATE_OUTPUT_PATH}')

## 1. Extract

The Git repository contains the compact client target table, Road Network reference, AusRAP road segments and the authoritative 49-site workbook. The 147 macro-enabled survey workbooks remain in approved archival storage outside Git. Therefore, this notebook can fully reproduce schema, identifier, road/SLK and final-value checks, but it does not pretend to re-read archival workbooks that are not present.

In [ ]:
# Read only the project sources required for this audit.
model = pd.read_excel(
    MODEL_PATH,
    sheet_name='Model_Data',
    dtype={
        'site_id': 'string',
        'survey_batch_id': 'string',
        'source_location_id': 'string',
        'location_id': 'string',
        'road_no': 'string',
    },
)
client_targets = pd.read_csv(
    CLIENT_TARGET_PATH,
    dtype={'ROAD_NO': 'string', 'ID': 'string'},
)
road_network = pd.read_csv(ROAD_NETWORK_PATH, low_memory=False)
ausrap = pd.read_excel(AUSRAP_PATH, sheet_name=0)

# Normalise source headers without changing the source files.
for frame in (model, client_targets, road_network, ausrap):
    frame.columns = frame.columns.astype(str).str.strip()

source_inventory = pd.DataFrame(
    [
        {'source': 'Authoritative model workbook', 'path': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'rows': len(model), 'role': 'Reviewed load target'},
        {'source': 'Compact client target table', 'path': str(CLIENT_TARGET_PATH.relative_to(PROJECT_ROOT)), 'rows': len(client_targets), 'role': 'Target comparison'},
        {'source': 'Road Network', 'path': str(ROAD_NETWORK_PATH.relative_to(PROJECT_ROOT)), 'rows': len(road_network), 'role': 'Road and SLK validation'},
        {'source': 'AusRAP segments', 'path': str(AUSRAP_PATH.relative_to(PROJECT_ROOT)), 'rows': len(ausrap), 'role': 'Segment and coordinate linkage'},
        {'source': 'Archived survey workbooks', 'path': 'Approved archive outside Git', 'rows': np.nan, 'role': 'Raw target reconstruction when required'},
    ]
)
display(source_inventory)

## 2. Transform: schema and core quality checks

The model table is one row per surveyed site. Stable keys, road/SLK fields, WGS84 coordinates, two observed targets, source provenance and QA fields are required before modelling. The checks below do not change the authoritative workbook.

In [ ]:
EXPECTED_COLUMNS = [
    'site_id', 'survey_batch_id', 'source_location_id', 'location_id', 'road_no',
    'slk_km', 'road_name', 'survey_location_name', 'latitude', 'longitude',
    'coordinate_crs', 'coordinate_method', 'ausrap_segment_match_count',
    'n_survey_days', 'obs_peak_cross_ped_per_h', 'obs_peak_along_ped_per_h',
    'target_source', 'qa_status', 'qa_note',
]
NUMERIC_COLUMNS = [
    'slk_km', 'latitude', 'longitude', 'ausrap_segment_match_count',
    'n_survey_days', 'obs_peak_cross_ped_per_h', 'obs_peak_along_ped_per_h',
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(model.columns))
unexpected_columns = sorted(set(model.columns) - set(EXPECTED_COLUMNS))
for column in NUMERIC_COLUMNS:
    model[column] = pd.to_numeric(model[column], errors='coerce')

core_checks = pd.Series(
    {
        'row_count_is_49': len(model) == 49,
        'site_id_is_unique': model['site_id'].nunique(dropna=False) == 49,
        'location_id_is_unique': model['location_id'].nunique(dropna=False) == 49,
        'no_missing_required_columns': not missing_columns,
        'no_unexpected_columns': not unexpected_columns,
        'all_rows_ready': model['qa_status'].eq('ready').all(),
        'all_rows_use_epsg_4326': model['coordinate_crs'].eq('EPSG:4326').all(),
        'required_numeric_values_present': model[NUMERIC_COLUMNS].notna().all().all(),
        'latitude_range_valid': model['latitude'].between(-90, 90).all(),
        'longitude_range_valid': model['longitude'].between(-180, 180).all(),
    },
    name='passed',
)
display(core_checks.to_frame())

## 3. Transform: source comparison and reconciliation

Four issue cases affected five site rows. The comparison first joins the compact client values through `source_location_id`, then checks the reviewed values stored in the model workbook. Corrections are accepted only when raw-workbook summaries or road references provide agreeing evidence.

In [ ]:
client_comparison = client_targets[
    ['ID', 'ROAD_NO', 'SLK', 'Road', 'obs_peak_cross', 'obs_peak_along']
].rename(
    columns={
        'ID': 'source_location_id',
        'ROAD_NO': 'client_road_no',
        'SLK': 'client_slk_km',
        'Road': 'client_road_name',
        'obs_peak_cross': 'client_cross_ped_per_h',
        'obs_peak_along': 'client_along_ped_per_h',
    }
)
comparison = model.merge(client_comparison, on='source_location_id', how='left', validate='many_to_one')
flagged_site_ids = ['250341-04', '260335-31', '260335-32', '260335-38', '260335-40']
comparison_columns = [
    'site_id', 'source_location_id', 'road_no', 'client_road_no',
    'obs_peak_cross_ped_per_h', 'client_cross_ped_per_h',
    'obs_peak_along_ped_per_h', 'client_along_ped_per_h',
    'target_source', 'qa_status',
]
display(comparison.loc[comparison['site_id'].isin(flagged_site_ids), comparison_columns])

In [ ]:
reconciliation_log = pd.DataFrame(
    [
        {
            'issue_case': 'Two-day aggregation used instead of three days',
            'site_id': '250341-04',
            'original_issue': 'Compact table stored cross 125 and along 25.5.',
            'reviewed_change': 'Cross 87; along 17.3333.',
            'evidence': 'All three raw days and embedded hourly values agree; original values reproduce Thursday-Friday only.',
        },
        {
            'issue_case': 'Incorrect road number',
            'site_id': '260335-31 / 260335-32',
            'original_issue': 'Client identifier used M010 for Coolgardie Esperance Highway.',
            'reviewed_change': 'Use H010; retain original M010 source identifiers.',
            'evidence': 'Road Network places SLK 366.85 and 165.45 on H010; M010 is Chidlow York Road.',
        },
        {
            'issue_case': 'Special workbook layout interpreted twice',
            'site_id': '260335-38',
            'original_issue': 'Compact table stored along 29.6667.',
            'reviewed_change': 'Along 18.3333; cross remains 4.3333.',
            'evidence': 'Actual 15-minute series and embedded hourly values agree on 18.3333.',
        },
        {
            'issue_case': 'Site missing from compact table and filename locality error',
            'site_id': '260335-40',
            'original_issue': 'No compact-table row; filename referred to Norseman.',
            'reviewed_change': 'Cross 53.6667; along 66.3333; locality standardised to Northampton.',
            'evidence': 'Three raw workbooks and embedded hourly values agree; internal label states NWCH Northampton.',
        },
    ]
)
display(reconciliation_log)

## 4. Transform: road and SLK linkage

Road identifiers are checked against both Road Network and AusRAP interval tables. A valid site must have at least one containing interval for its standardised road number and SLK. Matching uses half-open intervals `[Start SLK, End SLK)` so a site at a shared boundary is assigned to the segment beginning there rather than counted twice. AusRAP match counts are also compared with the stored audit field.

In [ ]:
# Convert interval fields once before matching all sites.
road_network['ROAD'] = road_network['ROAD'].astype('string').str.strip()
road_network['START_SLK'] = pd.to_numeric(road_network['START_SLK'], errors='coerce')
road_network['END_SLK'] = pd.to_numeric(road_network['END_SLK'], errors='coerce')
ausrap['Road No'] = ausrap['Road No'].astype('string').str.strip()
ausrap['Start SLK'] = pd.to_numeric(ausrap['Start SLK'], errors='coerce')
ausrap['End SLK'] = pd.to_numeric(ausrap['End SLK'], errors='coerce')


def count_interval_matches(frame: pd.DataFrame, road_column: str, start_column: str, end_column: str, road_no: str, slk_km: float) -> int:
    # Use half-open intervals to avoid double-counting shared segment boundaries.
    matches = frame.loc[
        frame[road_column].eq(road_no)
        & frame[start_column].le(slk_km)
        & frame[end_column].gt(slk_km)
    ]
    return int(len(matches))


road_audit = model[['site_id', 'road_no', 'slk_km', 'ausrap_segment_match_count']].copy()
road_audit['road_network_match_count'] = [
    count_interval_matches(road_network, 'ROAD', 'START_SLK', 'END_SLK', row.road_no, row.slk_km)
    for row in road_audit.itertuples(index=False)
]
road_audit['computed_ausrap_match_count'] = [
    count_interval_matches(ausrap, 'Road No', 'Start SLK', 'End SLK', row.road_no, row.slk_km)
    for row in road_audit.itertuples(index=False)
]
road_audit['stored_ausrap_count_matches'] = (
    road_audit['ausrap_segment_match_count'] == road_audit['computed_ausrap_match_count']
)
display(road_audit.loc[road_audit['site_id'].isin(flagged_site_ids)])

## 5. Load contract and final validation

After all validation checks pass, this notebook writes `model_dataset_candidate.xlsx` beside the notebook in `01_data_preparation/`. The candidate is a byte-for-byte copy of the validated authoritative workbook, so it preserves the `Metadata` and `Model_Data` sheets and their formatting for direct review. The approved source remains `data/processed/model_dataset.xlsx`; this notebook never replaces it. Because the archived survey workbooks are not in Git, this Load step packages the validated current result rather than claiming a complete rebuild from all raw surveys.

In [ ]:
expected_reviewed_values = {
    '250341-04': {'road_no': 'H006', 'cross': 87.0, 'along': 17.3333333333333},
    '260335-31': {'road_no': 'H010', 'cross': 3.0, 'along': 8.33333333333333},
    '260335-32': {'road_no': 'H010', 'cross': 2.0, 'along': 12.3333333333333},
    '260335-38': {'road_no': 'H044', 'cross': 4.33333333333333, 'along': 18.3333333333333},
    '260335-40': {'road_no': 'H007', 'cross': 53.6666666666667, 'along': 66.3333333333333},
}
reviewed_value_checks = {}
for site_id, expected in expected_reviewed_values.items():
    row = model.loc[model['site_id'].eq(site_id)].iloc[0]
    reviewed_value_checks[f'{site_id}_road'] = row['road_no'] == expected['road_no']
    reviewed_value_checks[f'{site_id}_cross'] = np.isclose(row['obs_peak_cross_ped_per_h'], expected['cross'])
    reviewed_value_checks[f'{site_id}_along'] = np.isclose(row['obs_peak_along_ped_per_h'], expected['along'])

final_checks = pd.concat(
    [
        core_checks,
        pd.Series(
            {
                'all_sites_match_road_network': road_audit['road_network_match_count'].gt(0).all(),
                'all_sites_match_ausrap': road_audit['computed_ausrap_match_count'].gt(0).all(),
                'stored_ausrap_counts_reproduced': road_audit['stored_ausrap_count_matches'].all(),
                'compact_table_missing_site_is_documented': comparison.loc[comparison['site_id'].eq('260335-40'), 'client_road_no'].isna().all(),
                'all_reviewed_values_match_the_audit_record': all(reviewed_value_checks.values()),
                'authoritative_workbook_was_not_overwritten': sha256_file(MODEL_PATH) == MODEL_SHA256_BEFORE,
            },
            name='passed',
        ),
    ]
)
assert final_checks.all(), final_checks.loc[~final_checks]

# Create the review workbook only after every source and quality check passes.
shutil.copy2(MODEL_PATH, CANDIDATE_OUTPUT_PATH)
load_checks = pd.Series(
    {
        'candidate_workbook_created': CANDIDATE_OUTPUT_PATH.is_file(),
        'candidate_matches_validated_workbook': sha256_file(CANDIDATE_OUTPUT_PATH) == sha256_file(MODEL_PATH),
        'authoritative_workbook_still_unchanged': sha256_file(MODEL_PATH) == MODEL_SHA256_BEFORE,
    },
    name='passed',
)
final_checks = pd.concat([final_checks, load_checks])
assert final_checks.all(), final_checks.loc[~final_checks]
display(final_checks.to_frame())
print(f'Audit complete: 49 unique sites passed, and the candidate workbook was written to {CANDIDATE_OUTPUT_PATH}.')

## Interpretation

The generated `model_dataset_candidate.xlsx` is suitable for direct review because all 49 site rows pass the documented QA checks and its hash matches the validated authoritative workbook. Final model reporting should still include a sensitivity analysis excluding `250341-04` and `260335-38`, because those two sites contain corrected target values. A complete raw-target rebuild remains possible only when the archived survey workbooks are mounted from approved storage.